# Tutorial 3: predicting gene expression from H&E morphology with a Vision Transformer

**ASI-FIMSA Workshop 2026, spatial omics, hands-on**

A pathologist can look at an H&E slide and tell you a great deal: this is tumour, that is stroma,
there is a lymphocyte cluster. If a *person* can read biology off morphology, can a *model* read it
off morphology too, quantitatively, gene by gene?

We take a public **10x Visium** breast cancer section, cut out the little square of H&E image
sitting under each of the 3,798 spots, and train a **Vision Transformer (ViT)** from scratch to
predict the expression of 16 breast and immune marker genes from that image patch alone. The task
was established by [**ST-Net**](https://www.nature.com/articles/s41551-020-0578-x) (2020) and
[**HE2RNA**](https://www.nature.com/articles/s41467-020-17678-4) (2020) using convolutional
networks; we use a Transformer, in the spirit of
[**HisToGene**](https://github.com/maxpmx/HisToGene).

You will come away with a plain-PyTorch Transformer you can read line by line, a Vision Transformer
built on top of it and trained end to end in about five minutes on a free CPU runtime, and an answer
to "which genes can you predict from a picture?" whose immunology punchline is that **the sparse
immune markers are exactly the ones this approach fails on**.

Everything runs in Colab, with the data downloaded live from 10x Genomics.

> Derived from `Deep_learning_04_vit_HE_spatial.ipynb` in the `gml-teaching-2026` course.

## 0. Setup

### 0.1 Install

Colab already ships **PyTorch**, **numpy**, **pandas**, **matplotlib**, **scipy**,
**scikit-learn**, **Pillow** and **tifffile**, so the only thing missing is `scanpy`, which we use
to read the 10x count matrix. This takes a minute or two.

In [ ]:
%pip install -q scanpy

### 0.2 Download the data

We use the public 10x Visium sample **`V1_Breast_Cancer_Block_A_Section_1`**, an invasive ductal
carcinoma section with 3,798 spots under tissue, processed with Space Ranger 1.1.0.

| File | Size | What it is |
| --- | --- | --- |
| `..._filtered_feature_bc_matrix.h5` | 28 MB | the counts: genes &times; spots |
| `..._spatial.tar.gz` | 10 MB | the spot coordinates, and a 2000 &times; 2000 px thumbnail |
| `..._image.tif` | **1.8 GB** | the full-resolution H&E, what the model learns from |

Why the big one is worth it: `tissue_hires_image.png` in the bundle is a 2000 &times; 2000 downsample
of a slide that is really **24,240 &times; 24,240**.

| | pixels per tile | µm per pixel | an 8 µm nucleus is |
| --- | --- | --- | --- |
| `tissue_hires_image.png` | 32, stretched up to 64 | 3.76 | **2 px** |
| `..._image.tif` | 390, shrunk down to 64 | 0.31 | 26 px |

This Tutorial asks what can be inferred *from morphology*, and on the thumbnail a lymphocyte is two
grey pixels. Both routes hand the model the same 64 &times; 64 tensor; only one fills it with real
tissue. If the TIFF download fails, the Tutorial falls back to the thumbnail and everything still
runs, just blurrier.

In [ ]:
%%bash
set -euo pipefail
BASE=https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1
SAMPLE=V1_Breast_Cancer_Block_A_Section_1

mkdir -p visium && cd visium

# The two small files. Both are required.
for f in ${SAMPLE}_filtered_feature_bc_matrix.h5 ${SAMPLE}_spatial.tar.gz; do
    if [ ! -s "$f" ]; then
        echo "downloading $f ..."
        wget -q --show-progress --tries=5 --timeout=60 --continue "$BASE/$f"
    else
        echo "$f already present, skipping"
    fi
done

# The full-resolution H&E, 1.8 GB. --continue makes this a no-op once the file is
# complete and resumes it if a previous attempt was cut short. The `|| echo` matters:
# without it a failed download would abort the whole cell under `set -e`, whereas we
# would rather carry on and fall back to the thumbnail.
TIF=${SAMPLE}_image.tif
echo "fetching $TIF (1.8 GB, resumable) ..."
wget -q --show-progress --tries=3 --timeout=120 --continue "$BASE/$TIF" \
    || echo "WARNING: full-resolution H&E did not download - falling back to the hi-res PNG"

tar -xzf ${SAMPLE}_spatial.tar.gz
echo
echo "--- visium/ ---";        ls -lh
echo "--- visium/spatial/ ---"; ls -lh spatial/

### 0.3 Imports and device

`device` is chosen at run time. If you have a GPU runtime the notebook will use it; on the default
CPU runtime it just runs a bit slower, and we cap PyTorch to 4 threads so it plays nicely with
Colab's small CPU allocation.

Results **differ slightly between CPU and GPU**, and between machines, even with the seed fixed,
because floating-point addition is not associative and the two backends add things up in different
orders. Your numbers will not match the presenter's to the third decimal. Look for stable
*conclusions* rather than bit-exact determinism.

In [ ]:
import warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from PIL import Image
import tifffile
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(4)          # keep CPU usage modest on a shared Colab runtime

print("torch     :", torch.__version__)
print("numpy     :", np.__version__)
print("scanpy    :", sc.__version__)
print("device    :", device)

SAMPLE   = "V1_Breast_Cancer_Block_A_Section_1"
DATA_DIR = Path("visium")
SPATIAL  = DATA_DIR / "spatial"
FULLRES_TIF = DATA_DIR / f"{SAMPLE}_image.tif"   # the 1.8 GB H&E

## 1. Load the three things we need

A Space Ranger output bundle contains a lot; we need exactly three pieces:

| Piece | File | Role here |
| --- | --- | --- |
| **Expression** | `filtered_feature_bc_matrix.h5` | what we are trying to **predict** (the target, `y`) |
| **The image** | `spatial/tissue_hires_image.png` | what we predict **from** (the input, `X`) |
| **Where each spot sits** | `spatial/tissue_positions*.csv` + `scalefactors_json.json` | how to line the two up |

The third row is the fiddly one. Space Ranger 1.1.0 wrote the spot coordinate table as
`tissue_positions_list.csv` with **no header row**; from Space Ranger 2.0 it is
`tissue_positions.csv` *with* one. `scanpy.read_visium()` handles either, so if you use it you never
see this. Read the CSV by hand, as our source notebook did and as a lot of code in the wild does, and
pandas eats the first spot as a header row. It fails silently. So we write a tiny loader that copes
with both conventions and **assert** that the answer makes sense afterwards.

In [ ]:
# The Space Ranger column order, which is fixed across versions even when the header is not.
POS_COLS = ["barcode", "in_tissue", "array_row", "array_col",
            "pxl_row_in_fullres", "pxl_col_in_fullres"]


def read_tissue_positions(spatial_dir):
    """Read the Visium spot table from either Space Ranger convention.

    Handles `tissue_positions_list.csv` (<= SR 1.x, no header) and
    `tissue_positions.csv` (>= SR 2.0, with header). Returns a DataFrame indexed by
    barcode with unambiguous `row_fullres` / `col_fullres` columns.
    """
    spatial_dir = Path(spatial_dir)
    for name in ("tissue_positions.csv", "tissue_positions_list.csv"):
        path = spatial_dir / name
        if path.exists():
            break
    else:
        raise FileNotFoundError(f"no tissue positions file found in {spatial_dir}")

    # Sniff: does the first line look like a header, or like data?
    with open(path) as fh:
        first_line = fh.readline()
    has_header = "barcode" in first_line.lower()

    if has_header:
        pos = pd.read_csv(path)
    else:
        pos = pd.read_csv(path, header=None, names=POS_COLS)

    print(f"read {path.name}  |  header row present: {has_header}  |  {len(pos)} spots")

    # Rename to names we control, so nothing downstream can confuse row with column.
    pos = pos.rename(columns={"pxl_row_in_fullres": "row_fullres",     # y, vertical
                              "pxl_col_in_fullres": "col_fullres"})    # x, horizontal
    return pos.set_index("barcode")


pos_all = read_tissue_positions(SPATIAL)
print(pos_all.head(3))
print("\nspots under tissue (in_tissue == 1):", int((pos_all.in_tissue == 1).sum()))

### 1.1 Expression

Standard scanpy preprocessing: drop genes seen in almost no spots and spots with almost no genes,
normalise every spot to the same total count, and `log1p` transform.

In [ ]:
adata = sc.read_10x_h5(DATA_DIR / f"{SAMPLE}_filtered_feature_bc_matrix.h5")
adata.var_names_make_unique()
print("as loaded          :", adata.shape, "(spots x genes)")

sc.pp.filter_genes(adata, min_cells=10)
sc.pp.filter_cells(adata, min_genes=200)
print("after filtering    :", adata.shape)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print("normalised + log1p")

### 1.2 Line the spots up with the image

Spot coordinates are recorded in **full-resolution** pixels, the coordinate system of the big TIFF,
which is exactly the frame we want for cutting tiles. `scalefactors_json.json` tells us how to shrink
them onto the 2000 px thumbnail for drawing an overview.

**A convention worth internalising:** we store the coordinates once, in `adata.obsm["spatial"]`, as
**(x, y)**, the order every spatial package in the Python ecosystem expects, and the *opposite* of
the (row, column) order the CSV uses. Fixing the convention in one place and then only ever reading
`adata.obsm["spatial"]` is how you avoid transposed-tissue plots.

Two pixel frames, both derived from `obsm["spatial"]`: `col_f`, `row_f` in **full-resolution** px
with no scale factor at all, used to cut tiles; and `col_h`, `row_h`, the same points times
`tissue_hires_scalef`, for drawing on the thumbnail.

In [ ]:
sf = json.load(open(SPATIAL / "scalefactors_json.json"))
hires_scalef = sf["tissue_hires_scalef"]
print("scalefactors:", sf)

# keep only in-tissue spots that also survived expression filtering
pos = pos_all[pos_all.in_tissue == 1]
common = [b for b in adata.obs_names if b in pos.index]
adata = adata[common].copy()
pos = pos.loc[common]

# THE canonical coordinate store: (x, y) in full-resolution pixels.
# Note this is built in `adata.obs_names` order, NOT the order of the positions file.
# Tiles are cut in the same order below, which is what keeps every tile matched to the
# right spot's expression. Get that wrong and the model trains happily to r = 0.
adata.obsm["spatial"] = pos[["col_fullres", "row_fullres"]].values.astype(float)

# --- frame 1: the thumbnail, for overview plots ----------------------------------
img = np.asarray(Image.open(SPATIAL / "tissue_hires_image.png").convert("RGB"))

xy_hires = adata.obsm["spatial"] * hires_scalef
col_h = xy_hires[:, 0].astype(int)      # x
row_h = xy_hires[:, 1].astype(int)      # y

# --- frame 2: the full-resolution H&E, for tiles ---------------------------------
# No scale factor: obsm["spatial"] is already in this frame.
col_f = adata.obsm["spatial"][:, 0].astype(int)      # x, full-res px
row_f = adata.obsm["spatial"][:, 1].astype(int)      # y, full-res px

he, USE_FULLRES = None, False
if FULLRES_TIF.exists():
    try:
        # Memory-map, do not read. This touches the header only; no pixels move yet.
        he = tifffile.memmap(FULLRES_TIF)
        USE_FULLRES = True
    except Exception as exc:                      # truncated or partial download
        print(f"!! {FULLRES_TIF.name} is present but unreadable ({exc})")

print("\nspots            :", adata.n_obs)
print("H&E thumbnail    :", img.shape, "(rows, cols, RGB)")
if USE_FULLRES:
    print("H&E full-res     :", he.shape, he.dtype, "  <- tiles come from here")
else:
    print("H&E full-res     : NOT AVAILABLE - falling back to the thumbnail for tiles")

print(f"\nspot x spans     : {col_h.min()} - {col_h.max()}  (thumbnail is {img.shape[1]} wide)")
print(f"spot y spans     : {row_h.min()} - {row_h.max()}  (thumbnail is {img.shape[0]} tall)")

# Sanity check: every spot must land inside the image, and the spots must actually
# cover a decent chunk of it. If the row/col convention were flipped this would fail.
# np.ptp(...) as a function, not .ptp() as a method: numpy 2.0 removed the method.
def check_frame(x, y, shape, name):
    assert (x >= 0).all() and (x < shape[1]).all(), f"{name}: x coordinates off the image"
    assert (y >= 0).all() and (y < shape[0]).all(), f"{name}: y coordinates off the image"
    assert np.ptp(x) > 0.3 * shape[1] and np.ptp(y) > 0.3 * shape[0], \
        f"{name}: spots cover too little of the image - check the row/col convention"
    print(f"  {name}: OK")

print("\ncoordinate checks")
check_frame(col_h, row_h, img.shape, "thumbnail frame")
if USE_FULLRES:
    check_frame(col_f, row_f, he.shape, "full-res frame ")

In [ ]:
# Look at the slide, with the spots drawn on top. This is the THUMBNAIL:
# never hand matplotlib the 24240 x 24240 array, it would try to rasterise
# all 1.8 GB of it. Overviews use the small image, tiles use the big one.
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(img)
axes[0].set_title(f"H&E thumbnail {img.shape[1]} x {img.shape[0]}")
axes[1].imshow(img)
axes[1].scatter(col_h, row_h, s=2, c="tab:cyan", alpha=0.5)
axes[1].set_title(f"{adata.n_obs} Visium spots on the same image")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Cut one H&E tile per spot

For every spot we crop a small square of image centred on it. That square, the **tile**, is the
model's entire view of the world for that spot.

**How big should the tile be?** Big enough to show the spot *and* a little of what surrounds it,
small enough that we are not mostly training on a neighbour's tissue. We derive the size from the
data rather than hard-coding a number:

```
WIN = spot_diameter_fullres x 2.2      # in full-resolution pixels
```

The factor 2.2 means "the spot plus roughly half a spot of context on every side". Deriving it
matters: the notebook this is based on used a different sample and hard-coded `WIN = 40`, which here
would be a quarter of a single spot.

Every tile is then resized to a fixed **64 &times; 64**. Note which way that resize runs: we shrink
390 px down to 64, throwing detail away. From the thumbnail we would be stretching 32 px *up* to 64,
inventing detail that was never recorded.

Two implementation details matter. We **memory-map the TIFF** rather than reading it, so a 1.8 GB
file costs one tile of RAM at a time, which is the ordinary way to work with whole-slide images. And
we resize with **`LANCZOS`, not `BILINEAR`**, because shrinking ~6&times; with a filter that does not
average over the pixels it discards would alias away the detail we downloaded 1.8 GB to get.

In [ ]:
OUT = 64                                  # tile size fed to the ViT

spot_hires_px = sf["spot_diameter_fullres"] * hires_scalef
win_hires     = int(round(spot_hires_px * 2.2))
win_fullres   = int(round(sf["spot_diameter_fullres"] * 2.2))

WIN = win_fullres if USE_FULLRES else win_hires   # crop window, in the frame we tile from

print(f"spot_diameter_fullres = {sf['spot_diameter_fullres']:.2f} full-res px")
print(f"tissue_hires_scalef   = {hires_scalef:.8f}")
print()
print(f"  on the thumbnail  a spot is {spot_hires_px:6.1f} px across"
      f"  ->  2.2x window = {win_hires:3d} px")
print(f"  on the full-res   a spot is {sf['spot_diameter_fullres']:6.1f} px across"
      f"  ->  2.2x window = {win_fullres:3d} px")
print()
frame = "FULL-RESOLUTION TIFF" if USE_FULLRES else "thumbnail (FALLBACK - tiles will be blurry)"
print(f"tiling from the {frame}")
print(f"  WIN = {WIN} px  ->  resized to {OUT} x {OUT}"
      f"   ({'shrinking' if WIN > OUT else 'stretching'} {max(WIN, OUT) / min(WIN, OUT):.1f}x)")

In [ ]:
# `he` is a memory-map: slicing it reads only the pages that slice touches, so the
# 1.8 GB file never lands in RAM. Do NOT np.pad() it -- padding a 24240 x 24240
# array would materialise 1.9 GB.
src, rr, cc = (he, row_f, col_f) if USE_FULLRES else (img, row_h, col_h)

tiles = np.zeros((adata.n_obs, OUT, OUT, 3), dtype=np.float32)
half = WIN // 2
for i, (r, c) in enumerate(zip(rr, cc)):
    r0, c0 = max(0, r - half), max(0, c - half)
    crop = np.asarray(src[r0:r0 + WIN, c0:c0 + WIN])
    if crop.shape[:2] != (WIN, WIN):          # only for spots close to an edge
        crop = np.pad(crop, ((0, WIN - crop.shape[0]),
                             (0, WIN - crop.shape[1]), (0, 0)), mode="edge")
    # LANCZOS, not BILINEAR: we are shrinking ~6x, and a filter that fails to average
    # over the pixels it drops would alias away the detail we came for.
    tiles[i] = np.asarray(Image.fromarray(crop).resize((OUT, OUT), Image.LANCZOS)) / 255.0

print("tiles:", tiles.shape, "  (spots, height, width, RGB)   values in",
      f"[{tiles.min():.2f}, {tiles.max():.2f}]")
assert tiles.shape == (adata.n_obs, OUT, OUT, 3)

In [ ]:
# A few example tiles. This is genuinely all the model ever sees.
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.ravel(), rng.choice(adata.n_obs, 16, replace=False)):
    ax.imshow(tiles[i])
    ax.axis("off")
fig.suptitle("Example H&E tiles - one per Visium spot")
plt.tight_layout()
plt.show()

## 3. Choose the genes to predict

We could ask the model to predict all 19,000 surviving genes, but that would be slow and almost
impossible to interpret. Instead we pick a panel of **16 interpretable markers**, chosen so the
answer will *mean* something to an immunologist:

| Genes | Compartment | Prior expectation |
| --- | --- | --- |
| `ERBB2`, `ESR1`, `KRT8`, `KRT18`, `EPCAM` | tumour epithelium | should be predictable, epithelial nests are visually obvious |
| `COL1A1`, `DCN`, `LUM` | fibroblasts / stroma | should be very predictable, collagen is the most distinctive thing on an H&E |
| `PTPRC` (CD45), `CD3D`, `CD68`, `MS4A1` (CD20) | pan-immune, T cell, macrophage, B cell | **the interesting ones**, see below |

Then we top the panel up to 16 with **highly variable genes**, so the model also has some targets it
was not hand-picked for.

Targets are the `log1p`-normalised expression, **z-scored per gene**. Without that, the loss would
be dominated by whichever gene happens to be most abundant and the model would ignore the rest.

In [ ]:
markers = ["ERBB2", "ESR1", "KRT8", "KRT18", "EPCAM",     # tumour epithelium
           "COL1A1", "DCN", "LUM",                        # stroma / fibroblast
           "PTPRC", "CD3D", "CD68", "MS4A1"]              # immune: pan, T, macrophage, B

present = [g for g in markers if g in adata.var_names]
missing = [g for g in markers if g not in adata.var_names]
print("markers found  :", present)
print("markers missing:", missing if missing else "none")

# top up to 16 targets with the most variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
for g in adata.var.sort_values("dispersions_norm", ascending=False).index:
    if len(present) >= 16:
        break
    if g not in present:
        present.append(g)
present = present[:16]
print("\n16 target genes:", present)

Yexpr = adata[:, present].X
Yexpr = np.asarray(Yexpr.todense()) if hasattr(Yexpr, "todense") else np.asarray(Yexpr)

ymean, ystd = Yexpr.mean(0), Yexpr.std(0) + 1e-8
Yz = (Yexpr - ymean) / ystd               # z-scored targets, shape (spots, 16)
print("target matrix  :", Yz.shape)

### 3.1 How sparse is each marker?

Before we train anything, look at how often each gene is detected at all. This table tells you most
of what the accuracy chart at the end will say.

A Visium spot is 55 µm across and holds roughly 1 to 10 cells. A gene expressed strongly by *every*
cell is detected in essentially every spot. A gene expressed by a rare cell type, at low copy
number, is detected in a minority of spots and is mostly **zeros**, and zeros carry no signal for
the model to learn from. This is **dropout**, and it hits exactly the immune markers we care most
about.

In [ ]:
detect = (Yexpr > 0).mean(0) * 100
tbl = pd.DataFrame({"gene": present,
                    "detected in % of spots": np.round(detect, 1),
                    "mean log-expression": np.round(Yexpr.mean(0), 2)})
print(tbl.sort_values("detected in % of spots", ascending=False).to_string(index=False))

In [ ]:
# The measured spatial pattern of one gene per compartment.
xy = np.c_[col_h, row_h]
show = [g for g in ["KRT8", "COL1A1", "PTPRC", "CD3D"] if g in present]

fig, axes = plt.subplots(1, len(show), figsize=(3.6 * len(show), 4))
for ax, g in zip(np.atleast_1d(axes), show):
    j = present.index(g)
    sca = ax.scatter(xy[:, 0], -xy[:, 1], c=Yexpr[:, j], cmap="magma", s=6)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(g)
    plt.colorbar(sca, ax=ax, shrink=0.7)
plt.suptitle("Measured expression in tissue space  (note how patchy the immune markers are)")
plt.tight_layout()
plt.show()

---

## 4. What is a Transformer, actually?

Before we build the Vision Transformer, the intuition. This section has no biology in it, but every
idea here shows up in scGPT, Geneformer, ESM, AlphaFold's Evoformer and every chatbot you have used.

### 4.1 A token is just "one thing, represented as a vector"

A Transformer does not know about images, or genes, or English. It consumes a **sequence of
tokens**, and a token is a **list of numbers** standing for one unit of the input. What counts as a
unit is your choice: a word for a language model, one gene in one cell for scGPT, one amino acid for
a protein model, and for a Vision Transformer **one small square patch of an image**.

So "a Transformer treats an image as a sentence" is almost literal: chop the tile into an
8 &times; 8 grid of patches, turn each patch into a vector, and you have a 64-token "sentence".

### 4.2 Self-attention: every token asks every other token a question

This is the one idea that makes a Transformer a Transformer. A convolution only ever looks at
neighbouring pixels; an RNN reads strictly left to right. Self-attention removes that constraint.
**Every token can look directly at every other token, and learns how much to care about each one.**

Each token emits three vectors: a **query** ("what I am looking for"), a **key** ("what I am,
advertised to others") and a **value** ("what I hand over if you attend to me"). For a given token,
compare its query against every token's key; high match means high relevance. Those scores go
through a softmax so they are positive and sum to 1, becoming **attention weights**, and the token's
output is the weighted average of everybody's values.

Two consequences worth holding on to. **Attention is learned, not wired in**: nobody told the model
that adjacent patches are related. And **attention weights are readable**, a genuine number per
token pair, which is why we can draw the attention map in Section 8.

**Multi-head** attention runs several query/key/value systems in parallel (`nhead=4` here) and
concatenates the results, so one head might track colour and another texture.

### 4.3 The `[CLS]` token

Self-attention updates each of the 64 patch tokens, but we want **one** prediction for the whole
tile. Averaging the 64 works, and forces every patch to contribute equally. The Transformer trick is
to **prepend one extra token that is not part of the data at all**, called `[CLS]`: it starts as a
learnable vector and carries no information about any particular patch.

Because attention lets every token see every other, `[CLS]` reads all 64 patches and learns how much
to weight each one. Its final vector is a learned summary of the tile, and that is what feeds the
prediction head. Reading the attention weights out of `[CLS]` is what Section 8 does.

### 4.4 Position: attention is blind to order, so we have to tell it

Self-attention computes a weighted average over a **set**. Shuffle the tokens and every output is
shuffled identically; the maths does not change, so the model has no idea which patch was top-left.

For a picture of tissue architecture that is unacceptable, so we add a learnable **positional
embedding**: a distinct vector per position, added to each token, learned along with everything else.

### 4.5 The block: attention, then a small MLP, both with a bypass

A Transformer **block** wraps the attention in two pieces of standard machinery:

```
        x ──────────────────────────┐  (residual "bypass" connection)
        │                           │
        ├── multi-head self-attention│
        │                           ▼
        └──────────────────────►  add ──► LayerNorm ──┐
                                                      │
        ┌─────────────────────────────────────────────┤
        │                                             │
        ├── feed-forward MLP (Linear → GELU → Linear) │
        │                                             ▼
        └────────────────────────────────────────► add ──► LayerNorm ──► out
```

The **residual connection** means each sub-layer only has to learn a *correction* to what came in;
without these, deep stacks do not train. **LayerNorm** keeps training stable. The **feed-forward
MLP** is applied to each token independently: attention lets tokens *exchange* information, the MLP
lets each token *process* what it received.

Stack the block a few times (`depth=3` here) and by layer 3 a patch has indirectly seen the entire
tile. That is the whole architecture.

In [ ]:
class TransformerBlock(nn.Module):
    """One Transformer block: multi-head self-attention, then a small MLP.

    Both sub-layers are wrapped as `LayerNorm(x + sublayer(x))` -- the residual
    connection plus normalisation drawn in the diagram above.
    """

    def __init__(self, d_model, nhead, dim_ff, dropout=0.1):
        super().__init__()
        # PyTorch gives us the attention primitive; we wire the block ourselves so
        # the data flow stays visible (and so we can read the attention weights out).
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, need_weights=False):
        # x is [batch, sequence, d_model]. Query, key and value are all x -- this is
        # what "SELF-attention" means: the sequence attends to itself.
        a, w = self.attn(x, x, x, need_weights=need_weights, average_attn_weights=True)
        x = self.norm1(x + self.drop(a))     # residual + norm  (attention)
        x = self.norm2(x + self.ff(x))       # residual + norm  (feed-forward)
        return x, w                          # w = attention weights, for interpretation


print(TransformerBlock(d_model=64, nhead=4, dim_ff=128))

### 4.6 A 30-second demonstration

Push a toy sequence of 5 random tokens through one untrained block. Two things to check:

1. The **output has exactly the same shape as the input**. A block transforms tokens, it does not
   change how many there are, which is why you can stack blocks freely.
2. The **attention matrix is 5 &times; 5**: one weight for every (token asking, token answering)
   pair, and each row sums to 1 because of the softmax.

In [ ]:
demo_block = TransformerBlock(d_model=64, nhead=4, dim_ff=128)
demo_block.eval()

demo_x = torch.randn(1, 5, 64)                     # 1 sequence, 5 tokens, 64 numbers each
with torch.no_grad():
    demo_out, demo_w = demo_block(demo_x, need_weights=True)

print("input  :", tuple(demo_x.shape),   " (batch, tokens, d_model)")
print("output :", tuple(demo_out.shape), " <- same shape, so blocks stack")
print("attention:", tuple(demo_w.shape), " (batch, token asking, token answering)")
print("\nattention weights (untrained, so nearly uniform):")
print(np.round(demo_w[0].numpy(), 3))
print("\nevery row sums to:", np.round(demo_w[0].sum(-1).numpy(), 3))

## 5. From Transformer to **Vision** Transformer

Section 4 was about a generic sequence of tokens. To do vision we need one extra piece: a way to turn
a **picture** into a **sequence**. That is the entire content of the
[Vision Transformer paper](https://arxiv.org/abs/2010.11929) (Dosovitskiy et al., 2021).

### 5.1 Patch embedding

> **Cut the image into a grid of small square patches. Each patch is a token.**

Our tile is 64 &times; 64 pixels. With `patch=8` we cut it into an 8 &times; 8 grid, so **64 patches**
and 64 tokens. Each patch (8 &times; 8 &times; 3 = 192 raw numbers) is projected down to a
`d_model = 64`-dimensional vector, which a **strided convolution** with kernel 8 and stride 8 does in
one line.

```
   64 x 64 x 3 tile
        │
        │  Conv2d(3 -> 64, kernel=8, stride=8)
        ▼
    8 x 8 x 64        →  flatten  →   64 tokens x 64 numbers
    (patch grid)                       ("a sentence of 64 patches")
        │
        │  prepend [CLS], add positional embedding
        ▼
    65 tokens x 64 numbers   →  3 x TransformerBlock  →  take token 0 ([CLS])
                                                              │
                                                    Linear(64 -> 16)
                                                              ▼
                                                 16 predicted gene values
```

Compare that with the sequence-of-genes view used by scGPT: the machinery in the middle is
*identical*, and only the tokeniser at the front differs.

### 5.2 The full model

Small on purpose: `d_model=64`, `nhead=4`, `depth=3`, about 118,000 parameters. Real ViTs are
1,000&times; larger and trained on millions of images. Ours trains from scratch on 2,848 tiles in a
few minutes on a laptop CPU, and still learns something real.

In [ ]:
class ViT(nn.Module):
    """A small Vision Transformer that regresses gene expression from an H&E tile."""

    def __init__(self, img=64, patch=8, d_model=64, nhead=4, depth=3, dim_ff=128, n_out=16):
        super().__init__()
        self.n_patch = (img // patch) ** 2                       # 8 x 8 = 64 patch tokens

        # 1. patch embedding: a non-overlapping strided conv = "cut into patches, embed each"
        self.proj = nn.Conv2d(3, d_model, kernel_size=patch, stride=patch)

        # 2. the [CLS] summary token, and one positional vector per sequence slot
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, self.n_patch + 1, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)

        # 3. the stack of Transformer blocks from Section 4 -- unchanged
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, nhead, dim_ff) for _ in range(depth)])

        # 4. the regression head: [CLS] vector -> 16 gene values
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_out)

    def forward(self, x, need_weights=False):
        x = self.proj(x).flatten(2).transpose(1, 2)     # [B, 3, 64, 64] -> [B, 64 patches, d]
        cls = self.cls.expand(x.size(0), -1, -1)        # one [CLS] per item in the batch
        x = torch.cat([cls, x], dim=1) + self.pos       # [B, 65, d], now position-aware

        attns = []
        for blk in self.blocks:
            x, w = blk(x, need_weights=need_weights)
            attns.append(w)

        return self.head(self.norm(x[:, 0])), attns     # predict from token 0 = [CLS]


demo_model = ViT(n_out=len(present))
with torch.no_grad():
    demo_pred, _ = demo_model(torch.tensor(tiles[:4]).permute(0, 3, 1, 2))
print("4 tiles in  ->", tuple(demo_pred.shape), "predictions out (4 spots x 16 genes)")
print("patch tokens per tile:", demo_model.n_patch, "(+1 for [CLS] =",
      demo_model.n_patch + 1, "tokens)")
print("trainable parameters :", sum(p.numel() for p in demo_model.parameters()))

## 6. Train

Ordinary supervised regression:

* **Input**: tiles as `[batch, 3, 64, 64]` tensors. PyTorch wants **channels first**, whereas images
  are normally stored channels-last, hence the `permute`.
* **Target**: the 16 z-scored marker values per spot. **Loss**: mean squared error.
* **Split**: 75% of spots to train, 25% held out for validation.
* **Metric**: the mean **Pearson correlation** between predicted and measured expression on the
  held-out spots. Correlation is right here because we care whether the model gets the *pattern*
  right across tissue, not whether it nails the absolute value.

> We split spots at random, which means a validation spot often has a training spot as its immediate
> neighbour, sharing tissue and some of the same image. That inflates the score relative to a
> completely unseen section. It is the standard shortcut in this literature, it keeps the *relative*
> ranking of genes meaningful, and a spatial split is the first thing you should fix in real work.

About 35 epochs at ~5 to 10 seconds each: roughly **3 to 6 minutes on a Colab CPU**, or well under a
minute on a GPU.

In [ ]:
Xt = torch.tensor(tiles).permute(0, 3, 1, 2).contiguous()      # [N, 3, 64, 64], channels-first
Yt = torch.tensor(Yz, dtype=torch.float32)

idx = np.arange(adata.n_obs)
tr, te = train_test_split(idx, test_size=0.25, random_state=SEED)
print(f"train spots: {len(tr)}   validation spots: {len(te)}")

model = ViT(n_out=len(present)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
print("trainable parameters:", sum(p.numel() for p in model.parameters()))


def predict(ids, batch=512):
    """Run the model over a set of spots and return a plain numpy array."""
    model.eval()
    out = []
    with torch.no_grad():
        for s in range(0, len(ids), batch):
            xb = Xt[ids[s:s + batch]].to(device)
            out.append(model(xb)[0].detach().cpu().numpy())
    return np.concatenate(out, axis=0)

In [ ]:
EPOCHS, BS = 35, 64
hist = {"loss": [], "mean_r": []}

for ep in range(EPOCHS):
    model.train()
    perm = np.random.permutation(tr)
    ep_loss = 0.0
    for s in range(0, len(perm), BS):
        b = perm[s:s + BS]
        xb = Xt[b].to(device)                  # move the batch to the device, not the lot
        yb = Yt[b].to(device)
        opt.zero_grad()
        out, _ = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * len(b)

    pred = predict(te)
    rs = [pearsonr(pred[:, j], Yz[te, j])[0] for j in range(len(present))]
    hist["loss"].append(ep_loss / len(tr))
    hist["mean_r"].append(np.nanmean(rs))

    if ep == 0 or (ep + 1) % 5 == 0:
        print(f"epoch {ep + 1:2d}   train MSE {hist['loss'][-1]:.3f}   "
              f"val mean r {hist['mean_r'][-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist["loss"], color="tab:red")
ax[0].set_title("Training loss (MSE)"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["mean_r"], color="tab:blue")
ax[1].axhline(0, ls="--", c="grey", lw=1)
ax[1].set_title("Validation mean Pearson r"); ax[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

Read those two curves together. The loss should fall smoothly, and the validation correlation should
rise and then flatten. If the loss keeps falling while the correlation *drops*, the model has started
memorising the training tiles, which is what you would expect if you pushed the epochs much higher
with a model this small and a dataset this size.

## 7. Which genes are predictable from morphology?

Here is the scientifically interesting result, and the reason this Tutorial exists. The mean
correlation across all 16 genes is not the point. The **spread** is. Look at which genes sit at the
top of the chart and which sit at the bottom, and ask whether that ordering makes biological sense.

In [ ]:
pred = predict(te)
rs = {present[j]: pearsonr(pred[:, j], Yz[te, j])[0] for j in range(len(present))}
order = sorted(rs, key=rs.get)

immune = {"PTPRC", "CD3D", "CD68", "MS4A1"}
colors = ["tab:red" if g in immune else "tab:green" for g in order]

plt.figure(figsize=(6.5, 5))
plt.barh(order, [rs[g] for g in order], color=colors)
plt.axvline(0, c="grey", lw=1)
plt.xlabel("Pearson r  (predicted vs measured, held-out spots)")
plt.title(f"Per-gene accuracy from H&E alone   |   mean r = {np.nanmean(list(rs.values())):.2f}\n"
          "red = immune markers", fontsize=10)
plt.tight_layout()
plt.show()

print("best predicted :", order[-3:][::-1])
print("worst predicted:", order[:3])

### 7.1 The immunology talking point

`PTPRC` (CD45), `CD3D`, `CD68` and `MS4A1` (CD20) should cluster near the bottom, while the stromal
(`COL1A1`, `DCN`, `LUM`) and epithelial (`KRT8`, `KRT18`, `EPCAM`) genes sit at the top. Why?

**The structural genes work** because collagen-rich stroma and epithelial tumour nests look
*completely different* on H&E: pink fibrillar matrix against dense sheets of large nuclei. A
pathologist separates them at a glance, and so can a 118,000-parameter network. Those genes are also
expressed at high level by the dominant cell type in the spot, so the measurement is stable.

**The immune genes fail** for three reasons that stack up:

1. **Dropout.** `MS4A1` is detected in a small minority of spots, and you cannot learn a mapping
   onto a column that is mostly zeros.
2. **Resolution.** Three T cells among seven tumour cells produce a spot that is transcriptionally
   *mostly tumour*.
3. **Morphological ambiguity.** A small round blue cell on H&E could be a T cell, a B cell, an NK
   cell, or a plasmacytoid dendritic cell. **The information is not in the image**, and no amount of
   training data or model capacity fixes that. This is what immunohistochemistry and multiplex
   imaging exist to resolve.

**The take-home.** Predicting expression from H&E is real and useful, and can extend sparse
molecular measurements across a whole slide cheaply. But it works best precisely where you needed it
least, on obvious structural compartments, and worst where you needed it most, on immune
infiltration. Any paper claiming to read the immune microenvironment off H&E deserves this chart
pointed at it.

> **Roughly what to expect** (your numbers will differ in the second decimal). Mean r across the
> panel around **0.32**, with `DCN`, `KRT8`, `COL1A1`, `ESR1`, `KRT18` and `LUM` at the top around
> 0.40 to 0.48, and all four hand-picked immune markers in the bottom third: `PTPRC` and `CD68` near
> 0.30, `CD3D` near 0.25, `MS4A1` near 0.19.

In [ ]:
# Predicted vs measured for the two best and the two worst genes.
picks = order[::-1][:2] + order[:2]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, g in zip(axes, picks):
    j = present.index(g)
    ax.scatter(Yz[te, j], pred[:, j], s=8, alpha=0.4,
               c="tab:red" if g in immune else "tab:green")
    lim = [-3, 4]
    ax.plot(lim, lim, ls="--", c="grey", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("measured (z)"); ax.set_ylabel("predicted (z)")
    ax.set_title(f"{g}   (r = {rs[g]:.2f})", fontsize=10)
plt.suptitle("Two best-predicted genes (left) and two worst (right)", y=1.04)
plt.tight_layout()
plt.show()

Notice the shape of the failures. For a poorly predicted gene the cloud collapses towards a
horizontal band: the model has given up and is predicting roughly the mean for every spot, which is
the optimal thing to do under MSE when you have no usable signal. That flat band is what "the
information is not in the image" looks like in a scatter plot.

### 7.2 Put it back on the tissue

Correlation coefficients are abstract. Painting predicted and measured expression side by side on
the slide is what makes it land.

In [ ]:
pred_all = predict(np.arange(adata.n_obs))
best, worst = order[-1], order[0]

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for row, g in enumerate([best, worst]):
    j = present.index(g)
    for col, (vals, ttl) in enumerate([(Yz[:, j], f"measured  {g}"),
                                       (pred_all[:, j], f"ViT-predicted  {g}")]):
        ax = axes[row, col]
        sca = ax.scatter(xy[:, 0], -xy[:, 1], c=vals, cmap="magma", s=6, vmin=-2, vmax=2)
        ax.set_aspect("equal"); ax.axis("off")
        ax.set_title(f"{ttl}   (r = {rs[g]:.2f})", fontsize=10)
        plt.colorbar(sca, ax=ax, shrink=0.7)
plt.suptitle("Top: the best-predicted gene.   Bottom: the worst.\n"
             "Same colour scale throughout.", y=1.0)
plt.tight_layout()
plt.show()

The top row should be strikingly similar: the model reconstructs the spatial architecture of the
tissue from morphology alone, having never been told where anything is. The bottom row should be
strikingly *dis*similar, a patchy measured map against a nearly flat prediction.

## 8. Where is the model looking? (attention rollout)

Because attention weights are just numbers we can read, we can ask the model to show its work:
**which patches did `[CLS]` actually listen to?**

Taking the attention weights from the last block alone is misleading, because those tokens have
already been mixed by the earlier blocks. **Attention rollout**
([Abnar & Zuidema, 2020](https://arxiv.org/abs/2005.00928)) composes the layers properly: add the
identity matrix to each layer's attention to account for the residual connection, re-normalise each
row to sum to 1, then matrix-multiply the layers together.

Take the `[CLS]` row of the result, drop the `[CLS]`-to-itself entry, reshape the remaining 64
numbers back to the 8 &times; 8 patch grid, and you have a heat map over the tile. Treat it as a
*hint* rather than a proof: rollout tells you where information flowed, which is related to but not
the same as what the model relied on causally.

In [ ]:
def attention_rollout(att_list):
    """Compose per-layer attention into one [CLS] -> patch relevance map."""
    A = torch.stack(att_list, 0)                       # [depth, batch, seq, seq]
    eye = torch.eye(A.size(-1), device=A.device)       # device-aware: works on CPU and GPU
    roll = None
    for d in range(A.size(0)):
        a = A[d] + eye                                 # residual connection
        a = a / a.sum(-1, keepdim=True)                # renormalise
        roll = a if roll is None else torch.bmm(a, roll)
    return roll[:, 0, 1:]                              # [CLS] row, patches only

In [ ]:
n_show = 6
ex = te[:n_show]
model.eval()
with torch.no_grad():
    _, attns = model(Xt[ex].to(device), need_weights=True)
maps = attention_rollout(attns).detach().cpu().numpy()

side = int(round(model.n_patch ** 0.5))               # 8
scale = tiles.shape[1] // side                        # 64 / 8 = 8 px per patch

fig, axes = plt.subplots(2, n_show, figsize=(2.2 * n_show, 4.8))
for k, i in enumerate(ex):
    axes[0, k].imshow(tiles[i]); axes[0, k].set_xticks([]); axes[0, k].set_yticks([])
    m = np.kron(maps[k].reshape(side, side), np.ones((scale, scale)))   # upscale to tile size
    axes[1, k].imshow(tiles[i])
    axes[1, k].imshow(m, cmap="jet", alpha=0.45)
    axes[1, k].set_xticks([]); axes[1, k].set_yticks([])
axes[0, 0].set_ylabel("H&E tile", fontsize=10)
axes[1, 0].set_ylabel("attention", fontsize=10)
plt.suptitle("What the [CLS] token attends to  (red = high, blue = low)")
plt.tight_layout()
plt.show()

With only 3 layers, 64-dimensional tokens and 2,848 training tiles, do not expect textbook saliency
maps. What you should see is that attention is **not uniform**: the model has learned to weight some
regions of the tile above others, typically the nucleus-dense areas rather than empty white space.

---

## Further reading

Our ViT is trained from scratch on 2,848 tiles. Production methods start from a **pathology
foundation model** (UNI, CONCH, Prov-GigaPath, Virchow), ViTs pretrained on hundreds of thousands of
whole-slide images and then fine-tuned with a new head for exactly this task. Because you have now
built a ViT by hand, you know precisely what those models are doing inside.

### Where this Tutorial sits

Tutorial 1 read the data; Tutorial 2 asked which cells sit next to which; Tutorial 2b asked what
might be passing between them; this Tutorial asked what you can infer from morphology alone.

The through-line across all four: **every one of these methods has a boundary, and the boundary
usually lands on the immune compartment.** Tutorial 2 could not resolve a B cell cluster. Tutorial 2b
ranks every immune pair below the extracellular matrix. This Tutorial predicts stromal genes from
morphology and immune genes badly. Three different methods, three different failure modes, one
consistent blind spot, and knowing where it lands is the difference between using these tools and
being used by them.